## Microsoft Agent Framework (MAF): Ramp-Up Training Materials

To get the latest version of MAF Python package, use:

``` bash
pip install --upgrade agent-framework --pre
pip install --upgrade azure-ai-projects --pre
```

## 📒 Notebook 1: MCP Hosted Tool

### 🪜 Step 1: Configure environment

In [1]:
# Suppress warnings from agent_framework_azure module
import warnings
warnings.filterwarnings(
    action = "ignore",
    category = UserWarning,
    module = "agent_framework_azure"
)

In [2]:
# Import required packages
import os
import asyncio
from agent_framework import HostedMCPTool
from agent_framework.azure import AzureAIClient
from azure.identity.aio import DefaultAzureCredential

In [3]:
# Fix: Disable Accept-Encoding to prevent compressed responses
import azure.core.pipeline.policies as policies

# Store original
_original_on_request = policies.HeadersPolicy.on_request

def patched_on_request(self, request):
    """Remove Accept-Encoding header to prevent compressed responses."""
    _original_on_request(self, request)
    # Tell server we don't accept compressed responses
    request.http_request.headers['Accept-Encoding'] = 'identity'

policies.HeadersPolicy.on_request = patched_on_request
print("Applied Accept-Encoding fix to disable compression")

Applied Accept-Encoding fix to disable compression


In [4]:
# Set environment variables
PROJECT_ENDPOINT = os.environ.get("AZURE_FOUNDRY_PROJECT_ENDPOINT")
MODEL_DEPLOYMENT = os.environ.get("AZURE_FOUNDRY_GPT_MODEL")

if not PROJECT_ENDPOINT or not MODEL_DEPLOYMENT:
    print("WARNING: Environment variables not set properly!")
else:
    print("Environment variables set successfully!")

Environment variables set successfully!


### 🪜 Step 2: Define MCP tool

In [5]:
# Define MCP tool
mcp_doc_tool = HostedMCPTool(
    name = "microsoft-learn-mcp-tool",
    url = "https://learn.microsoft.com/api/mcp",
    approval_mode = "never_require"
)

print(f"Created MCP tool: {mcp_doc_tool.name}")

Created MCP tool: microsoft-learn-mcp-tool


### 🪜 Step 3: Create AI agent

In [6]:
# Define AI client
ai_client = AzureAIClient(
    agent_name = "MAFSDK-Agentv2-MCP",
    project_endpoint = PROJECT_ENDPOINT,
    model_deployment_name = MODEL_DEPLOYMENT,
    async_credential = DefaultAzureCredential()
)

print(f"Created AI client: {ai_client.agent_name}")

Created AI client: MAFSDK-Agentv2-MCP


In [7]:
# Create agent with MCP tool
agent = ai_client.create_agent(
    name = "microsoft-documentation-agent",
    instructions = "You are an agent, which can use its MCP documentation tool to answer end user questions about Microsoft products. Limit your response to 2 paragraphs.",
    tools = [mcp_doc_tool]
)

print(f"Created MS Doc agent: {agent.name}, equipped with MCP tool.")

Created MS Doc agent: microsoft-documentation-agent, equipped with MCP tool.


### 🪜 Step 4: Run the agent

In [8]:
# Test chat agent with a prompt
prompt = "How can I create a new Azure AI Foundry resource?"
print(f"User:\n{prompt}\n")

print(f"Agent:")
async for update in agent.run_stream(prompt):
    if update.text:
        print(update.text, end="", flush=True)

User:
How can I create a new Azure AI Foundry resource?

Agent:
You can create a new Azure AI Foundry resource using the Azure portal, Azure CLI, or Azure PowerShell. In the Azure portal, select the Foundry resource creation link https://portal.azure.com/#create/Microsoft.CognitiveServicesAIFoundry, then provide subscription, resource group, region, and a name for the resource. After configuring any additional settings and accepting terms, you can review and create the resource. The Foundry resource supports managing multiple projects and deployments for generative AI models and applications.

Alternatively, you can create the Foundry resource via CLI with the command:
```
az cognitiveservices account create --name <resource-name> --resource-group <resource-group> --kind AIServices --sku S0 --location <region> --yes
```
or via PowerShell with the command:
```
New-AzCognitiveServicesAccount -ResourceGroupName <resource-group> -Name <resource-name> -Type AIServices -SkuName S0 -Location 

### 🪜 Step 5: Housekeeping

In [9]:
# Delete Azure AI client
await ai_client.close()

print("AI client closed successfully.")

AI client closed successfully.
